# Deploy Vela 2.0 0.3B on a CPU endpoint on SageMaker AI

Deploy [Vela 2.0 0.3B](https://huggingface.co/vllm-sr/Vela-2.0-0.3B) on one
`ml.c7i.2xlarge` real-time endpoint with no GPU. Invoke its five native question
types through Transformers `AutoModel` and `system_one`: Choice, Yes/No (`noul`),
Score, plus the Span and Set extensions that return labelled text offsets and
multi-label selections. The checkpoint and its remote engine code are pinned to the
same revision.

It runs on the maintained AWS PyTorch 2.14 **CPU** SageMaker DLC, through the small
serving image that this notebook builds with AWS CodeBuild. Read
[README.md](README.md) for prerequisites, licensing, and runtime details. This is
an evaluation example.

Run from this directory. Run the cleanup cell when finished, including after a
failed deployment once SageMaker allows endpoint deletion.

## 1. Environment preparation

In [ ]:
%pip install --upgrade 'boto3>=1.43.1,<2' --quiet

In [ ]:
import json
import math
import os
import statistics
import tarfile
import time
import uuid
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

session = boto3.Session()
region = session.region_name or "us-east-1"
sm = session.client("sagemaker", region_name=region)
runtime = session.client(
    "sagemaker-runtime", region_name=region,
    config=Config(read_timeout=70, retries={"max_attempts": 0}),
)
s3 = session.client("s3", region_name=region)
iam = session.client("iam")
logs = session.client("logs", region_name=region)
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]

MODEL_ID = "vllm-sr/Vela-2.0-0.3B"
MODEL_REVISION = "d6f03aa9baca4f9017fa8f8e9c2764d3bed3f409"

# Set this when the notebook's own role is not a SageMaker execution role.
execution_role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")
bucket = os.environ.get("SAGEMAKER_ARTIFACT_BUCKET", f"sagemaker-{region}-{account_id}")
# A CPU instance. The FP32 worker peaks below 3 GiB; 8 vCPUs keep windowed long
# inputs well inside the 60-second InvokeEndpoint limit. Try ml.c7i.xlarge to go cheaper.
instance_type = os.environ.get("SAGEMAKER_INSTANCE_TYPE", "ml.c7i.2xlarge")
# Torch intra-op threads in the worker; empty uses every vCPU of the instance.
cpu_threads = os.environ.get("SAGEMAKER_CPU_THREADS", "")
# Set this to reuse a serving image you already built; otherwise section 2 builds one.
inference_image = os.environ.get("SAGEMAKER_INFERENCE_IMAGE_URI")
codebuild_role_arn = os.environ.get("CODEBUILD_ROLE_ARN")
cpu_base_image = os.environ.get(
    "SAGEMAKER_SERVING_BASE_IMAGE",
    "public.ecr.aws/deep-learning-containers/pytorch:2.14-cpu-amzn2023-sagemaker",
)
name = f"vela-2-0-3b-cpu-{uuid.uuid4().hex[:12]}"
artifact_key = f"vela-2-0-3b-cpu/{name}/model.tar.gz"
artifact_version = None
example_dir = Path.cwd()
archive_path = example_dir / f"{name}.tar.gz"
for filename in ["inference.py", "requirements.txt", "example-request.json"]:
    if not (example_dir / filename).is_file():
        raise FileNotFoundError(f"Run the notebook from its example directory: missing {filename}")
print(f"Region: {region}; model: {MODEL_ID}; instance: {instance_type}; endpoint: {name}")

### Resolve the execution role

In Studio the current role may already be appropriate. Outside Studio,
set `SAGEMAKER_EXECUTION_ROLE_ARN` to an existing SageMaker role. This cell
reads the role's actual ARN through IAM, preserving paths such as
`service-role/`, and checks its SageMaker trust.

In [ ]:
if execution_role is None:
    caller_arn = identity["Arn"]
    if ":assumed-role/" not in caller_arn:
        raise ValueError("Set SAGEMAKER_EXECUTION_ROLE_ARN to an existing SageMaker execution role.")
    role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
else:
    role_name = execution_role.rsplit("/", 1)[-1]
role_info = iam.get_role(RoleName=role_name)["Role"]
if execution_role is not None and execution_role != role_info["Arn"]:
    raise ValueError("The configured execution-role ARN does not match the resolved IAM role.")
execution_role = role_info["Arn"]
trusts_sagemaker = any(
    statement.get("Effect") == "Allow"
    and "sagemaker.amazonaws.com" in (
        [statement.get("Principal", {}).get("Service")]
        if isinstance(statement.get("Principal", {}).get("Service"), str)
        else statement.get("Principal", {}).get("Service", [])
    )
    for statement in role_info["AssumeRolePolicyDocument"]["Statement"]
)
if not trusts_sagemaker:
    raise ValueError(
        "The current role does not trust SageMaker. Set SAGEMAKER_EXECUTION_ROLE_ARN "
        "to an appropriate existing execution role and rerun setup."
    )
print("Execution role resolved and SageMaker trust verified.")

## 2. Build the CPU serving image

There is no AWS PyTorch inference DLC after 2.6, and that image is past its end of
support. This example runs on the maintained PyTorch 2.14 **CPU** SageMaker DLC through
the small shared image in
[03-features/decision-model-serving-image](../../../03-features/decision-model-serving-image/),
built with its `BASE_IMAGE` set to the CPU DLC. It adds `serve`, which answers `/ping`
and `/invocations` by calling this example's `model_fn` and `transform_fn`. The
[Decision 2.0 Kai 0.6B CPU example](../Decision-2.0-Kai-0.6B-cpu/) uses the same
image; reuse it with `SAGEMAKER_INFERENCE_IMAGE_URI`.

AWS CodeBuild builds the image and pushes it to a private Amazon ECR repository in your
account. Set `CODEBUILD_ROLE_ARN` to a CodeBuild service role that can read the artifact
bucket, write build logs, and push to Amazon ECR.

In [ ]:
import sys

if inference_image is None:
    if not codebuild_role_arn:
        raise ValueError("Set CODEBUILD_ROLE_ARN, or SAGEMAKER_INFERENCE_IMAGE_URI to reuse an image.")
    image_dir = example_dir.parents[2] / "03-features" / "decision-model-serving-image"
    if not image_dir.is_dir():
        raise FileNotFoundError(f"{image_dir} not found. Run this notebook from a clone of the repository.")
    sys.path.insert(0, str(image_dir))
    from build_image import build_image

    inference_image = build_image(
        region=region, bucket=bucket, codebuild_role_arn=codebuild_role_arn,
        base_image=cpu_base_image, tag_prefix="pt214-cpu",
    )
print(f"Serving image: {inference_image}")

## 3. Prepare the model artifact

Package the companion handler and requirements under `code/`, as expected by
the serving image. The requirements file installs nothing: PyTorch, Transformers,
and the Hugging Face libraries come from the image, and the Vela 2.0 engine
ships inside the pinned model repository as remote code. The worker downloads the
1.2 GB FP32 checkpoint, tokenizer, and calibration at startup; the ONNX exports in
the repository are not fetched. Review the published code before adopting it for
another workload.

The default artifact bucket is shared and is preserved during cleanup. Capture
the uploaded object's version directly from the upload response.

In [ ]:
try:
    s3.head_bucket(Bucket=bucket)
except ClientError as error:
    if error.response["Error"]["Code"] not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    create_args = {"Bucket": bucket}
    if region != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**create_args)
    s3.put_public_access_block(
        Bucket=bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True, "IgnorePublicAcls": True,
            "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
        },
    )

with tarfile.open(archive_path, "w:gz") as archive:
    for filename in ["inference.py", "requirements.txt"]:
        archive.add(example_dir / filename, arcname=f"code/{filename}")
with archive_path.open("rb") as artifact:
    uploaded = s3.put_object(Bucket=bucket, Key=artifact_key, Body=artifact)
artifact_version = uploaded.get("VersionId")
model_data_url = f"s3://{bucket}/{artifact_key}"
print("Inference artifact uploaded.")

## 4. Create the model, endpoint configuration, and endpoint

SageMaker allows 1,800 seconds for the container to pass its health check, which
covers pulling the image and downloading the checkpoint from Hugging Face. The
serving image runs one process, so the weights load once, and the adapter
serializes inference. Vela 2.0 loads in FP32; the model card does not support
BF16 or FP16 Torch loading.

`VELA_THREADS` sets Torch intra-op threads (default: every vCPU). No
`InferenceAmiVersion` or `VolumeSizeInGB` is set for a CPU instance.

In [ ]:
environment = {
    "SAGEMAKER_PROGRAM": "inference.py",
    "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
    "HF_HOME": "/tmp/huggingface",
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "TOKENIZERS_PARALLELISM": "false",
}
if cpu_threads:
    environment["VELA_THREADS"] = cpu_threads
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=execution_role,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataUrl": model_data_url,
        "Environment": environment,
    },
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": name,
        "InitialInstanceCount": 1, "InstanceType": instance_type,
        "InitialVariantWeight": 1.0,
        "ModelDataDownloadTimeoutInSeconds": 900,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print(f"Endpoint {name} is deploying.")

In [ ]:
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
endpoint = sm.describe_endpoint(EndpointName=name)
assert endpoint["EndpointStatus"] == "InService"
assert endpoint["EndpointConfigName"] == name
assert sum(v["CurrentInstanceCount"] for v in endpoint["ProductionVariants"]) == 1
print("Endpoint is InService with one instance.")

## 5. Verify the runtime and invoke typed questions

The adapter accepts `{"state": ..., "questions": {...}}` through SageMaker's
IAM-authenticated API and calls `model.system_one`. Its diagnostics operation
reports effective versions, the CPU model and thread settings, the ModernBERT
backbone geometry, the 8,192-token window and its overlap, the shipped
calibration, and the worker's peak resident memory.

In [ ]:
def invoke(payload):
    response = runtime.invoke_endpoint(
        EndpointName=name, ContentType="application/json",
        Accept="application/json", Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read())

diagnostics = invoke({"operation": "diagnostics"})
assert diagnostics["device"] == "cpu"
assert diagnostics["cuda_available"] is False
assert diagnostics["model"] == MODEL_ID
assert diagnostics["model_revision"] == MODEL_REVISION
assert diagnostics["loaded_parameters"] == 309114371
assert diagnostics["parameter_dtypes"] == {"torch.float32": 309114371}
assert diagnostics["backbone"]["model_type"] == "modernbert"
assert diagnostics["backbone"]["num_hidden_layers"] == 22
assert diagnostics["max_input_tokens"] == 8192
assert diagnostics["window_overlap"] == 512
assert diagnostics["versions"]["torch"].startswith("2.14.")
assert diagnostics["versions"]["transformers"] == "5.17.0"
assert diagnostics["torch_threads"] == int(cpu_threads or diagnostics["cpu_count"])
print(json.dumps(diagnostics, indent=2))

### PII spans, unsupported claims, and a route in one request

The state has typed parts: the user's `request`, a grounding `source`, and the
model's `answer`. Ask which spans of the request are personal information, which
spans of the answer the source does not support, and which subject area the
request belongs to. Span questions return labelled text with code-point offsets
into the targeted field, a `thresholds` entry, and a Yes/No view in `answers`.
The harness owns what happens with the result.

In [ ]:
request = json.loads((example_dir / "example-request.json").read_text())
result = invoke(request)
print(json.dumps(result, indent=2))

In [ ]:
assert result["model"] == diagnostics["model"]
answers, spans = result["answers"], result["spans"]
assert set(answers) == {"pii", "halu", "domain"}
assert set(spans) == {"pii", "halu"}
assert set(result["thresholds"]) == {"pii", "halu"}
for key in ("pii", "halu"):
    assert answers[key]["type"] == "noul"
    assert math.isfinite(answers[key]["noul"]) and 0 <= answers[key]["noul"] <= 1
    field = request["questions"][key].get("over", "answer")
    for span in spans[key]:
        assert set(span) == {"label", "start", "end", "text", "probability"}
        assert span["label"] in request["questions"][key]["criteria"]
        assert 0 <= span["start"] < span["end"] and 0 <= span["probability"] <= 1
        assert request["state"][field][span["start"]:span["end"]] == span["text"]
domain = answers["domain"]
assert domain["type"] == "choice"
assert domain["choice"] in request["questions"]["domain"]["criteria"]
probabilities = list(domain["probabilities"].values())
assert all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities)
assert abs(sum(probabilities) - 1) <= 0.001
print("Span and Choice outputs verified.")

### Set, Yes/No, and Score on a support ticket

A Set question scores every label independently and selects those above the
shipped (or supplied) threshold; each label also appears as a `noul` view
named `issues.<label>`. The Yes/No question supplies `true`/`false` criteria,
which sharpens Vela's Yes/No answers considerably (see the next section).

In [ ]:
ticket = invoke({
    "state": "My card was charged twice and the parcel never arrived.",
    "questions": {
        "issues": {
            "type": "set", "instructions": "Which issues does the customer report?",
            "criteria": {
                "billing": "payments, charges, refunds or invoices",
                "shipping": "delivery of an order or a parcel",
                "login": "signing in, passwords or account access",
            },
        },
        "billing": {
            "type": "noul", "instructions": "Is this request about billing?",
            "criteria": {
                "true": "the message concerns payments, charges, refunds or invoices",
                "false": "the message concerns anything else",
            },
        },
        "severity": {
            "type": "score", "instructions": "How severe is the issue?",
            "criteria": ["Minor", "Moderate", "Severe"],
        },
    },
})
print(json.dumps(ticket, indent=2))
issues = ticket["sets"]["issues"]
assert set(issues["probabilities"]) == {"billing", "shipping", "login"}
assert set(issues["selected"]) <= set(issues["probabilities"])
assert 0 <= ticket["thresholds"]["issues"] <= 1
assert set(ticket["answers"]) == {"issues.billing", "issues.shipping", "issues.login", "billing", "severity"}
assert ticket["answers"]["billing"]["type"] == "noul"
assert ticket["answers"]["severity"]["type"] == "score"
assert 0 <= ticket["answers"]["severity"]["score"] <= 2
print("Set, Yes/No, and Score outputs verified.")

### Basic expected-answer smoke checks

These simple cases check the serving path. They are not a representative
benchmark of decision quality or calibration.

The last check contrasts two formulations of the same question. A bare Yes/No
("Did the user explicitly provide a city?") separates the two states only
weakly on Vela, while a two-way Choice with concrete examples separates them
almost completely. Prefer one comparative Choice, or a Yes/No with explicit
`true`/`false` criteria, over bare Yes/No questions.

In [ ]:
assert domain["choice"] == "health"
assert {span["label"] for span in spans["pii"]} == {"PERSON", "EMAIL_ADDRESS"}
assert any(span["text"] == "up to 6 grams" for span in spans["halu"])
assert set(issues["selected"]) == {"billing", "shipping"}
assert ticket["answers"]["billing"]["noul"] >= 0.5

language = invoke({
    "state": "This is a clear sentence written in English.",
    "questions": {"language": {
        "type": "choice", "instructions": "Which language is this sentence written in?",
        "criteria": {"English": "English", "French": "French", "German": "German"},
    }},
})
assert language["answers"]["language"]["choice"] == "English"

city_choice = {"city": {
    "type": "choice", "instructions": "Does the request name the city for the weather lookup?",
    "criteria": {
        "named": "the request names a specific city or place, for example "
                 "'weather in Paris' or 'is it raining in Tokyo'",
        "missing": "the request asks about the weather without naming any city or place, "
                   "for example 'what is the weather' or 'will it rain today'",
    },
}}
city_noul = {"city": {"type": "noul", "instructions": "Did the user explicitly provide a city?"}}
explicit_city = "The user says: What is the current weather in Paris?"
missing_city = "The user says: What is the current weather?"
named = invoke({"state": explicit_city, "questions": city_choice})["answers"]["city"]
unnamed = invoke({"state": missing_city, "questions": city_choice})["answers"]["city"]
assert named["choice"] == "named" and unnamed["choice"] == "missing"
yes = invoke({"state": explicit_city, "questions": city_noul})["answers"]["city"]["noul"]
no = invoke({"state": missing_city, "questions": city_noul})["answers"]["city"]["noul"]
assert yes > no
print(f"Choice P(named): {named['probabilities']['named']:.3f} vs {unnamed['probabilities']['named']:.3f}; "
      f"bare Yes/No: {yes:.3f} vs {no:.3f}")
print("Basic expected-answer checks: 6/6.")

### Repeatability and server latency

Three repeats give a hosting smoke measurement, not a production latency
estimate. `latency_ms` measures server-side inference; SDK/network time is
measured separately in the scaling check. The example request runs two
sequences, because each sequence carries at most one Span question.

In [ ]:
repeats = [invoke(request) for _ in range(3)]
assert all(item["answers"] == result["answers"] and item["spans"] == result["spans"] for item in repeats)
print(f"Median warm server latency: {statistics.median(item['latency_ms'] for item in repeats):.2f} ms")

### Invalid questions and an oversized schema

The engine validates the request and reports the failing location, as the
published SystemOne server does. The adapter maps an invalid question to HTTP
422 and a schema that alone exceeds the 8,192-token window to HTTP 413.
SageMaker wraps both in `ModelError` and may omit the error body.

In [ ]:
def expect_model_error(payload, status):
    try:
        invoke(payload)
    except runtime.exceptions.ModelError as error:
        assert error.response.get("OriginalStatusCode") == status, error.response
        return
    raise AssertionError(f"Expected model HTTP {status}.")

expect_model_error({"state": "hello", "questions": {"q": {"type": "bogus", "instructions": "x"}}}, 422)
expect_model_error({
    "state": "hello",
    "questions": {"q": {
        "type": "choice", "instructions": "Pick one.",
        "criteria": {
            f"option_{index}": "lorem ipsum dolor sit amet consectetur adipiscing elit sed do eiusmod tempor " * 3
            for index in range(250)
        },
    }},
}, 413)
print("Invalid question rejected with HTTP 422; oversized schema with HTTP 413.")

### Long state is windowed, not rejected

Unlike the Decision 2.0 models, Vela scores a targeted part that does not fit the
window in overlapping 8,192-token windows and averages the logits, so a long
state is answered rather than rejected. `usage.input_tokens` then exceeds the
window size and latency grows with the number of windows. Each window is a full
8,192-token encoder pass, so keep this in mind when sizing: the request below
took 27 seconds on `ml.c7i.2xlarge` and 15 seconds on `ml.c7i.4xlarge`, and
must stay under SageMaker's 60-second invocation limit. A harness should bound state length itself.

In [ ]:
long_request = {"state": "weather city location " * 3000, "questions": city_choice}
# engine_stats counters are cumulative for the worker's lifetime; measure the delta.
windows_before = invoke({"operation": "diagnostics"})["engine_stats"].get("windows", 0)
started = time.perf_counter()
windowed = invoke(long_request)
round_trip_ms = (time.perf_counter() - started) * 1000
assert windowed["usage"]["input_tokens"] > diagnostics["max_input_tokens"]
assert windowed["answers"]["city"]["choice"] in city_choice["city"]["criteria"]
windows = invoke({"operation": "diagnostics"})["engine_stats"].get("windows", 0) - windows_before
assert windows >= 2, windows
print(f"{windowed['usage']['input_tokens']} input tokens in {windows} windows: "
      f"server {windowed['latency_ms']:.0f} ms, round trip {round_trip_ms:.0f} ms")

### Bounded multi-question scaling

Run 1, 8, 32, and 64 independent Yes/No question IDs over the same short state.
These are repeatability and hosting checks, not a decision-quality benchmark.
Non-span questions share one sequence, so the sequence grows with the schema
rather than repeating the state per question.

Measure three requests per question count. Server latency excludes SDK/network
time; round-trip latency includes it. Worker RSS is the lifetime high-water
mark, including startup and every preceding request, and does not include every
process on the hosting instance. These results apply only to this short input,
runtime, thread setting, and instance.

In [ ]:
scaling_results = []
for count in [1, 8, 32, 64]:
    payload = {
        "state": "The user explicitly requested the current weather in Paris.",
        "questions": {
            f"city_{index}": {
                "type": "noul",
                "instructions": "Did the user explicitly provide a city?",
            }
            for index in range(count)
        },
    }
    server_times, round_trip_times = [], []
    baseline_answers = None
    for _ in range(3):
        started = time.perf_counter()
        response = invoke(payload)
        round_trip_times.append((time.perf_counter() - started) * 1000)
        assert set(response["answers"]) == set(payload["questions"])
        for answer in response["answers"].values():
            assert answer["type"] == "noul" and math.isfinite(answer["noul"])
            assert 0 <= answer["noul"] <= 1
        if baseline_answers is None:
            baseline_answers = response["answers"]
        else:
            assert response["answers"] == baseline_answers
        server_times.append(response["latency_ms"])
    memory = invoke({"operation": "diagnostics"})
    scaling_results.append({
        "questions": count,
        "input_tokens": response["usage"]["input_tokens"],
        "server_median_ms": round(statistics.median(server_times), 2),
        "round_trip_median_ms": round(statistics.median(round_trip_times), 2),
        "worker_peak_rss_bytes": memory["worker_peak_rss_bytes"],
        "worker_peak_rss_scope": "worker_lifetime",
    })
print(json.dumps({
    "instance_type": instance_type,
    "processor": memory["processor"],
    "torch_threads": memory["torch_threads"],
    "results": scaling_results,
}, indent=2))

## 6. Cleanup

Delete this example's endpoint and wait before deleting its configuration and
model. Remove every S3 version and delete marker for this example's unique
artifact key, including repeated uploads. Preserve the shared bucket and role.
Attempt independent cleanup even if another operation fails; report failures.

The serving image, its Amazon ECR repository (`decision-model-serving`), and the
CodeBuild project are kept so later runs can reuse them; see the README to
remove them. Late endpoint log delivery can recreate the CloudWatch log group
after deletion. Rerun cleanup after delivery settles if the group reappears.

In [ ]:
def ignore_missing(operation, **arguments):
    try:
        return operation(**arguments)
    except ClientError as error:
        message = error.response["Error"].get("Message", "")
        error_code = error.response["Error"]["Code"]
        if error_code == "ResourceNotFoundException" or (
            error_code == "ValidationException"
            and ("Could not find" in message or "does not exist" in message)
        ):
            return None
        raise

def delete_endpoint():
    if ignore_missing(sm.describe_endpoint, EndpointName=name) is not None:
        sm.delete_endpoint(EndpointName=name)
        sm.get_waiter("endpoint_deleted").wait(
            EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
        )

def artifact_versions():
    return [
        {"Key": item["Key"], "VersionId": item["VersionId"]}
        for page in s3.get_paginator("list_object_versions").paginate(
            Bucket=bucket, Prefix=artifact_key
        )
        for group in ("Versions", "DeleteMarkers")
        for item in page.get(group, [])
        if item["Key"] == artifact_key
    ]

def delete_artifact():
    for item in artifact_versions():
        s3.delete_object(Bucket=bucket, **item)
    if artifact_versions():
        raise RuntimeError("S3 artifact versions remain after cleanup.")

cleanup_errors = []
def attempt_cleanup(label, operation, *positional, **arguments):
    try:
        operation(*positional, **arguments)
        return True
    except Exception as error:
        cleanup_errors.append(f"{label}: {error}")
        return False

try:
    if attempt_cleanup("endpoint", delete_endpoint):
        attempt_cleanup(
            "endpoint configuration", ignore_missing,
            sm.delete_endpoint_config, EndpointConfigName=name,
        )
        attempt_cleanup("model", ignore_missing, sm.delete_model, ModelName=name)
    attempt_cleanup("S3 artifact", delete_artifact)
    attempt_cleanup(
        "endpoint logs", ignore_missing,
        logs.delete_log_group, logGroupName=f"/aws/sagemaker/Endpoints/{name}",
    )
finally:
    attempt_cleanup("local archive", archive_path.unlink, missing_ok=True)

if cleanup_errors:
    raise RuntimeError("Cleanup incomplete: " + "; ".join(cleanup_errors))
assert ignore_missing(sm.describe_endpoint, EndpointName=name) is None
assert ignore_missing(sm.describe_endpoint_config, EndpointConfigName=name) is None
assert ignore_missing(sm.describe_model, ModelName=name) is None
print("Endpoint, configuration, model, all artifact versions, and endpoint logs cleaned up.")